# KQL performance demo – generate the test data

Creates a synthetic web shop clickstream in a Fabric Eventhouse (KQL database):

| Table | Rows | Content |
|---|---|---|
| `ShopEvents` | 100 million (default) | page views, searches, cart, checkout, purchases, errors over 90 days |
| `Users` | 1 million | user ID, segment, signup date, home country |
| `Products` | 1,000 | product ID, name, category, list price |

**How it works**

- The fact data is loaded in chunks, **one after another** (an Eventhouse accepts only one
  `.set-or-append` at a time – parallel commands are throttled).
- Every chunk covers its own time slice, so the data lands in time order, like a real stream.
  That's what makes time filters effective in the performance test.
- A **fixed start date** keeps the data identical no matter on which day or for how long the
  notebook runs.
- The notebook is **resumable**: chunks that are already complete are skipped, so you can simply
  run it again after an interruption.

**Before you start:** use a separate Eventhouse / KQL database for the demo and copy its
**Query URI** (Eventhouse → database → *Overview* → *Query URI*).

In [ ]:
# Install the Kusto client first – %pip restarts the Python session,
# so it has to run BEFORE the parameters are set.
%pip install azure-kusto-data --quiet

In [ ]:
# Parameters
KUSTO_URI      = "https://<your-eventhouse>.z0.kusto.fabric.microsoft.com"   # Query URI of the Eventhouse
DATABASE       = "eh_perf"                                                   # KQL database name

START_DATE     = "2026-08-18"   # first day of the data (UTC)
DAYS           = 90             # total time range
CHUNKS         = 10             # number of load commands (one time slice each)
ROWS_PER_CHUNK = 10_000_000     # 10 x 10 million = 100 million rows

USERS          = 1_000_000
PRODUCTS       = 1_000

DROP_EXISTING  = False          # True = drop the demo tables first and start from scratch

## 1. Connect to the Eventhouse

In [ ]:
import time
from datetime import datetime, timedelta

from azure.kusto.data import KustoClient, KustoConnectionStringBuilder, ClientRequestProperties
from azure.kusto.data.exceptions import KustoServiceError
from azure.kusto.data.helpers import dataframe_from_result_table


def _client():
    # A fresh token per call: loading 100 million rows takes longer than a token lives.
    token = notebookutils.credentials.getToken("kusto")
    kcsb = KustoConnectionStringBuilder.with_aad_user_token_authentication(KUSTO_URI, token)
    return KustoClient(kcsb)


def _props(minutes):
    props = ClientRequestProperties()
    props.set_option(ClientRequestProperties.request_timeout_option_name, timedelta(minutes=minutes))
    return props


TRANSIENT = ("connection", "transport", "network", "timed out", "timeout", "reset", "temporarily")


def _retry(call, what, retries=6):
    """Retries short calls on network hiccups (connection closed, timeouts)."""
    for attempt in range(1, retries + 1):
        try:
            return call()
        except Exception as error:
            if any(t in str(error).lower() for t in TRANSIENT) and attempt < retries:
                print(f"  network issue during {what} – retry {attempt}/{retries - 1} in {10 * attempt} s")
                time.sleep(10 * attempt)
            else:
                raise


def run_command(command, timeout_minutes=10):
    """Runs a short management command."""
    result = _retry(lambda: _client().execute_mgmt(DATABASE, command, _props(timeout_minutes)), "command")
    return dataframe_from_result_table(result.primary_results[0])


def run_query(query, timeout_minutes=10):
    """Runs a query and returns the first result table as a pandas DataFrame."""
    result = _retry(lambda: _client().execute_query(DATABASE, query, _props(timeout_minutes)), "query")
    return dataframe_from_result_table(result.primary_results[0])


def running_loads():
    """Number of .set-or-append operations still in progress in this database."""
    ops = run_command(".show operations")
    if ops.empty:
        return 0
    ops = ops[ops["Operation"] == "TableSetOrAppend"]
    if "Database" in ops.columns:
        ops = ops[ops["Database"] == DATABASE]
    latest = ops.sort_values("LastUpdatedOn").groupby("OperationId").tail(1)
    return int((latest["State"] == "InProgress").sum())


def wait_until_no_load_is_running(poll_seconds=20):
    while (n := running_loads()) > 0:
        print(f"  {n} load(s) still running – waiting {poll_seconds} s")
        time.sleep(poll_seconds)


def run_load(command, poll_seconds=15, max_resubmits=6):
    """Starts a long .set-or-append with async and polls until it has finished.
    Short calls only – no connection has to stay open for minutes."""
    async_command = command.replace(".set-or-append ", ".set-or-append async ", 1)
    for attempt in range(1, max_resubmits + 1):
        wait_until_no_load_is_running()
        op_id = run_command(async_command).iloc[0]["OperationId"]
        while True:
            time.sleep(poll_seconds)
            op = run_command(f".show operations {op_id}").sort_values("LastUpdatedOn").iloc[-1]
            state = op["State"]
            if state == "InProgress":
                continue
            if state == "Completed":
                return
            if state == "Throttled" and attempt < max_resubmits:
                print(f"  throttled – resubmitting in {30 * attempt} s")
                time.sleep(30 * attempt)
                break
            raise RuntimeError(f"Load {op_id} ended with state {state}: {op.get('Status', '')}")


print(run_query("print Database = current_database(), Now = now()"))

## 2. Tables

In [ ]:
if DROP_EXISTING:
    run_command(".drop tables (Purchases, ShopEvents, Users, Products) ifexists")
    print("Existing demo tables dropped.")

run_command("""
.create-merge table ShopEvents (
    Timestamp: datetime, EventTimeUnix: long, EventId: string, SessionId: string, UserId: string,
    EventType: string, Url: string, ProductId: int, Country: string, Payload: dynamic,
    Message: string, DurationMs: int)
""")
run_command(".create-merge table Users (UserId: string, Segment: string, SignupDate: datetime, HomeCountry: string)")
run_command(".create-merge table Products (ProductId: int, Name: string, Category: string, ListPrice: real)")
print("Tables ready.")

## 3. Dimension data (loaded only if the table is empty)

In [ ]:
def row_count(table):
    return int(run_query(f"{table} | count").iloc[0, 0])


if row_count("Products") == 0:
    run_load(f"""
.set-or-append Products <|
    range ProductId from 1 to {PRODUCTS} step 1
    | project ProductId = toint(ProductId),
              Name      = strcat("Product ", ProductId),
              Category  = tostring(dynamic(["Electronics", "Home", "Kitchen", "Sports", "Books",
                                            "Garden", "Toys", "Fashion", "Beauty", "Office"])[(ProductId - 1) % 10]),
              ListPrice = round(5 + rand() * 295, 2)
""")

if row_count("Users") == 0:
    run_load(f"""
.set-or-append Users <|
    range i from 0 to {USERS - 1} step 1
    | project UserId      = strcat("U", i),
              Segment     = tostring(dynamic(["New", "Regular", "Loyal", "VIP"])[toint(rand(4))]),
              SignupDate  = datetime({START_DATE}) - 3 * 365d + rand() * 3 * 365d,
              HomeCountry = tostring(dynamic(["DE", "AT", "CH", "NL", "FR", "BE", "PL", "DK", "IT", "ES"])[toint(rand(10))])
""")

print(f"Products: {row_count('Products'):,}   Users: {row_count('Users'):,}")

## 4. Fact data in chunks

Each chunk is checked first: complete chunks are skipped, empty chunks are loaded.
A partly filled chunk stops the notebook – that only happens if a load was interrupted;
the message tells you how to clean it up.

In [ ]:
CHUNK_TEMPLATE = """
.set-or-append ShopEvents <|
    range i from 1 to {n} step 1
    | extend k = {k}
    | extend Timestamp = datetime({chunk_start}) + {span} * (todouble(i - 1) / {n})
    | extend r = rand()
    | extend EventType = case(r < 0.70, "page_view",
                              r < 0.85, "search",
                              r < 0.93, "add_to_cart",
                              r < 0.97, "checkout",
                              r < 0.995, "purchase",
                                         "error")
    | extend UserId    = strcat("U", toint(rand({users}))),
             SessionId = strcat("S", k, "-", i / 20),
             ProductId = toint(rand({products}) + 1),
             IsMobile  = rand() < 0.6,
             OrderNo   = strcat("O", k, "-", i)
    | project
        Timestamp,
        EventTimeUnix = datetime_diff("millisecond", Timestamp, datetime(1970-01-01)),
        EventId       = tostring(new_guid()),
        SessionId,
        UserId,
        EventType,
        Url           = case(EventType == "purchase", strcat("https://shop.example.com/checkout/confirm?order=", OrderNo),
                             EventType == "search",   strcat("https://shop.example.com/search?q=item", ProductId),
                                                      strcat("https://shop.example.com/product/", ProductId)),
        ProductId,
        Country       = tostring(dynamic(["DE", "AT", "CH", "NL", "FR", "BE", "PL", "DK", "IT", "ES"])[toint(rand(10))]),
        Payload       = bag_pack(
                            "device",   iff(IsMobile, "mobile", "desktop"),
                            "browser",  tostring(dynamic(["Edge", "Chrome", "Firefox", "Safari"])[toint(rand(4))]),
                            "os",       iff(IsMobile, iff(rand() < 0.5, "Android", "iOS"), "Windows"),
                            "price",    round(5 + rand() * 295, 2),
                            "referrer", tostring(dynamic(["google", "newsletter", "instagram", "direct", "partner"])[toint(rand(5))])),
        Message       = case(EventType == "purchase", strcat("Order ", OrderNo, " confirmed for user ", UserId),
                             EventType == "error",    strcat("ERROR 504 timeout while loading product ", ProductId),
                                                      strcat("User ", UserId, " ", EventType, " product ", ProductId)),
        DurationMs    = toint(rand(3000))
"""

start = datetime.fromisoformat(START_DATE)
chunk_span = timedelta(days=DAYS) / CHUNKS
span_kql = f"{int(chunk_span.total_seconds())}s"     # e.g. 777600s = 9 days

for k in range(CHUNKS):
    chunk_start = start + k * chunk_span
    chunk_end = chunk_start + chunk_span
    fmt = "%Y-%m-%dT%H:%M:%S"
    wait_until_no_load_is_running()   # a load from an interrupted run may still be finishing
    existing = int(run_query(
        f"ShopEvents | where Timestamp >= datetime({chunk_start:{fmt}}) "
        f"and Timestamp < datetime({chunk_end:{fmt}}) | count").iloc[0, 0])

    label = f"Chunk {k + 1:>2}/{CHUNKS}  {chunk_start:%Y-%m-%d} – {chunk_end:%Y-%m-%d}"
    if existing == ROWS_PER_CHUNK:
        print(f"{label}: already complete, skipped")
        continue
    if existing > 0:
        raise RuntimeError(
            f"{label}: {existing:,} of {ROWS_PER_CHUNK:,} rows – partly loaded. Delete them with\n"
            f".delete table ShopEvents records <| ShopEvents | where Timestamp >= datetime({chunk_start:{fmt}}) "
            f"and Timestamp < datetime({chunk_end:{fmt}})\nand run this cell again.")

    t0 = time.time()
    run_load(CHUNK_TEMPLATE.format(n=ROWS_PER_CHUNK, k=k, chunk_start=f"{chunk_start:{fmt}}",
                                      span=span_kql, users=USERS, products=PRODUCTS))
    print(f"{label}: {ROWS_PER_CHUNK:,} rows loaded in {time.time() - t0:,.0f} s")

print("Fact data complete.")

## 5. Checks

In [ ]:
summary = run_query("""
ShopEvents
| summarize Rows = count() by Day = startofday(Timestamp)
| summarize Days = count(), MinPerDay = min(Rows), MaxPerDay = max(Rows), TotalRows = sum(Rows),
            From = format_datetime(min(Day), "yyyy-MM-dd"), To = format_datetime(max(Day), "yyyy-MM-dd")
""")
display(summary)

display(run_query("""
ShopEvents
| summarize Events = count() by EventType
| extend Share = strcat(round(100.0 * Events / toscalar(ShopEvents | count), 1), " %")
| order by Events desc
"""))

expected = CHUNKS * ROWS_PER_CHUNK
total = int(summary.loc[0, "TotalRows"])
print(f"Rows: {total:,} (expected {expected:,}) – {'OK' if total == expected else 'CHECK!'}")
print(f"Products: {row_count('Products'):,}   Users: {row_count('Users'):,}")

## 6. Cleanup (optional)

Run this cell only when the performance test is done – it deletes all demo tables.

In [ ]:
# run_command(".drop tables (Purchases, ShopEvents, Users, Products) ifexists")
# print("Demo tables dropped.")